In [2]:
class A:
    def __init__(self, name, old):
        super().__init__()
        self.name = name
        self.old = old


class B:
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)


class C(B, A):
    def __init__(self, name, old, weight, height):
        super().__init__(name, old)
        self.weight = weight
        self.height = height


person = C("Balakirev", 33, 80, 185)
person.__dict__

{'name': 'Balakirev', 'old': 33, 'weight': 80, 'height': 185}

# Подвиг 4.

С помощью множественного наследования удобно описывать принадлежность объектов к нескольким разным группам. Выполним такой пример.



Определите в программе классы в соответствии с их иерархией, представленной на рисунке выше:

Digit, Integer, Float, Positive, Negative

Каждый объект этих классов должен создаваться однотипной командой вида:

obj = Имя_класса(value)
где value - числовое значение. В каждом классе следует делать свою проверку на корректность значения value:

- в классе Digit: value - любое число;
- в классе Integer: value - целое число;
- в классе Float: value - вещественное число;
- в классе Positive: value - положительное число;
- в классе Negative: value - отрицательное число.

Если проверка не проходит, то генерируется исключение командой:

raise TypeError('значение не соответствует типу объекта')
После этого объявите следующие дочерние классы:

PrimeNumber - простые числа; наследуется от классов Integer и Positive;
FloatPositive - наследуется от классов Float и Positive.

Создайте три объекта класса PrimeNumber и пять объектов класса FloatPositive с произвольными допустимыми для них значениями. Сохраните все эти объекты в виде списка digits.

Затем, используя функции isinstance() и filter(), сформируйте следующие списки из указанных объектов:

lst_positive - все объекты, относящиеся к классу Positive;
lst_float - все объекты, относящиеся к классу Float.

P.S. В программе требуется объявить только классы и создать списки. На экран выводить ничего не нужно.

In [171]:
# Правильное решение

class Digit:
    def __init__(self, value):
        self.func = lambda x: isinstance(x, (int, float))
        self.execute_func(value)

    def execute_func(self, value):
        if not self.func(value):
            raise TypeError('значение не соответствует типу объекта')
        super().__setattr__('value', value)

class Integer(Digit):
    def __init__(self, value):
        self.func = lambda x: isinstance(x, int) # в случае с MRO это нельзя было определить как атрибут класса
        self.execute_func(value)
        super().__init__(value)

class Float(Digit):
    def __init__(self, value):
        self.func = lambda x: isinstance(x, float)
        self.execute_func(value)
        super().__init__(value)

class Positive(Digit):
    def __init__(self, value):
        self.func = lambda x: x>0
        self.execute_func(value)
        super().__init__(value)

class Negative(Digit):
    def __init__(self, value):
        self.func = lambda x: x<0
        self.execute_func(value)
        super().__init__(value)

class PrimeNumber(Integer, Positive):
    pass

class FloatPositive(Float, Positive):
    pass

digits = [PrimeNumber(3), PrimeNumber(1), PrimeNumber(4), FloatPositive(1.5), FloatPositive(9.2), FloatPositive(6.5),
          FloatPositive(3.5), FloatPositive(8.9)]

lst_positive = filter(lambda x: isinstance(x, Positive), digits)
lst_float  = filter(lambda x: isinstance(x, Float), digits)

==========================

In [175]:
class Digit:
    def __init__(self, value):
        self._value = value

    def __setattr__(self, name, value):
        if not self._check_value(value):
            raise TypeError('значение не соответствует типу объекта')
        super().__setattr__(name, value)

    def _check_value(self, value):
        return type(value) in (int, float)


class Integer(Digit):
    def _check_value(self, value):
        return super()._check_value(value) and type(value) is int


class Float(Digit):
    def _check_value(self, value):
        return super()._check_value(value) and type(value) is float


class Positive(Digit):
    def _check_value(self, value):
        return super()._check_value(value) and value > 0


class Negative(Digit):
    def _check_value(self, value):
        return super()._check_value(value) and value < 0


class PrimeNumber(Integer, Positive):
    pass


class FloatPositive(Float, Positive):
    pass


digits = [PrimeNumber(1), PrimeNumber(2), PrimeNumber(3),
          FloatPositive(1.2), FloatPositive(1.3), FloatPositive(1.4),
          FloatPositive(1.5), FloatPositive(1.6)]

lst_positive = list(filter(lambda x: isinstance(x, Positive), digits))
lst_float = list(filter(lambda x: isinstance(x, Float), digits))

# Подвиг 5.

В программе объявлены два класса:

class ShopItem:
    ID_SHOP_ITEM = 0

    def __init__(self):
        super().__init__()
        ShopItem.ID_SHOP_ITEM += 1
        self._id = ShopItem.ID_SHOP_ITEM

    def get_pk(self):
        return self._id


class Book(ShopItem):
    def __init__(self, title, author, year):
        super().__init__()
        self._title = title
        self._author = author
        self._year = year
Затем, создается объект класса Book (книга) и отображается в консоль:

book = Book("Python ООП", "Балакирев", 2022)
print(book)
В результате, на экране увидим что то вроде:

<__main__.Book object at 0x0000015FBA4B3D00>

Но нам требуется, чтобы здесь отображались локальные атрибуты объекта с их значениями в формате:

<атрибут_1>: <значение_1>
<атрибут_2>: <значение_2>
...
<атрибут_N>: <значение_N>

Для этого вам дают задание разработать два класса:

ShopGenericView - для отображения всех локальных атрибутов объектов любых дочерних классов (не только Book);
ShopUserView - для отображения всех локальных атрибутов, кроме атрибута _id, объектов любых дочерних классов (не только Book).

То есть, в этих классах нужно переопределить два магических метода: __str__() и __repr__().

Пример использования классов (эти строчки в программе писать не нужно):

class Book(ShopItem, ShopGenericView): ...
book = Book("Python ООП", "Балакирев", 2022)
print(book)
# на экране увидим строчки:
# _id: 1
# _title: Python ООП
# _author: Балакирев
# _year: 2022
Другой вариант использования классов:

class Book(ShopItem, ShopUserView): ...
book = Book("Python ООП", "Балакирев", 2022)
print(book)
# на экране увидим строчки:
# _title: Python ООП
# _author: Балакирев
# _year: 2022
P.S. В программе требуется объявить только классы. На экран выводить ничего не нужно.

In [185]:
class ShopItem:
    ID_SHOP_ITEM = 0

    def __init__(self):
        super().__init__()
        ShopItem.ID_SHOP_ITEM += 1
        self._id = ShopItem.ID_SHOP_ITEM

    def get_pk(self):
        return self._id


class ShopGenericView:
    '''для отображения всех локальных атрибутов объектов любых дочерних классов (не только Book);'''
    def __str__(self):
        return '\n'.join([f'{key}: {value}' for key, value in self.__dict__.items()])


class ShopUserView:
    '''для отображения всех локальных атрибутов, кроме атрибута _id, объектов любых дочерних классов (не только Book).'''
    def __str__(self):
        return '\n'.join([f'{key}: {value}' for key, value in self.__dict__.items() if not key == '_id'])

    # def __repr__(self):
    #     return self.__str__()


class Book(ShopItem, ShopUserView):
    def __init__(self, title, author, year):
        super().__init__()
        self._title = title
        self._author = author
        self._year = year

book = Book("Python ООП", "Балакирев", 2022)
print(book)

_title: Python ООП
_author: Балакирев
_year: 2022


In [186]:
book

# Подвиг 8 (введение в паттерн миксинов - mixins).

Часто множественное наследование используют для наполнения дочернего класса определенным функционалом. То есть, с указанием каждого нового базового класса, дочерний класс приобретает все больше и больше возможностей. И, наоборот, убирая часть базовых классов, дочерний класс теряет соответствующую часть функционала.

Например, паттерн миксинов активно используют в популярном фреймворке Django.  В частности, когда нужно указать дочернему классу, какие запросы от клиента он должен обрабатывать (запросы типа GET, POST, PUT, DELETE и т.п.). В качестве примера реализуем эту идею в очень упрощенном виде, но сохраняя суть паттерна миксинов.

Предположим, что в программе уже существует следующий набор классов:

class RetriveMixin:
    def get(self, request):
        return "GET: " + request.get('url')


class CreateMixin:
    def post(self, request):
        return "POST: " + request.get('url')


class UpdateMixin:
    def put(self, request):
        return "PUT: " + request.get('url')
Здесь в каждом классе выполняется имитация обработки запросов. За GET-запрос отвечает метод get() класса RetriveMixin, за POST-запрос - метод post() класса CreateMixin, за PUT-запрос - метод put() класса UpdateMixin.

Далее, вам нужно объявить класс с именем GeneralView, в котором следует указать атрибут (на уровне класса):

allowed_methods = ('GET', 'POST', 'PUT', 'DELETE')
для перечня разрешенных запросов. А также объявить метод render_request со следующей сигнатурой:

def render_request(self, request): ...

Здесь request - это словарь (объект запроса), в котором обязательно должны быть два ключа:

'url' - адрес для обработки запроса;
'method' - метод запроса: 'GET', 'POST', 'PUT', 'DELETE' и т. д.

В методе render_request() нужно сначала проверить, является ли указанный запрос в словаре request разрешенным (присутствует в списке allowed_methods). И если это не так, то генерировать исключение командой:

raise TypeError(f"Метод {request.get('method')} не разрешен.")
Иначе, вызвать метод по его имени:

method_request = request.get('method').lower()  # имя метода, малыми буквами
Подсказка: чтобы получить ссылку на метод с именем method_request, воспользуйтесь функцией getattr().

Для использования полученных классов, в программе объявляется следующий дочерний класс:

class DetailView(RetriveMixin, GeneralView):
    allowed_methods = ('GET', 'PUT', )
Воспользоваться им можно, например, следующим образом (эти строчки в программе не писать):

view = DetailView()
html = view.render_request({'url': 'https://stepik.org/course/116336/', 'method': 'GET'})
print(html)   # GET: https://stepik.org/course/116336/
Если в запросе указать другой метод:

html = view.render_request({'url': 'https://stepik.org/course/116336/', 'method': 'PUT'})
то естественным образом возникнет исключение (реализовывать в программе не нужно, это уже встроено в сам язык Python):

AttributeError: 'DetailView' object has no attribute 'put'

так как дочерний класс DetailView не имеет метода put. Поправить это можно, если указать соответствующий базовый класс:

class DetailView(RetriveMixin, UpdateMixin, GeneralView):
    allowed_methods = ('GET', 'PUT', )
Теперь, при выполнении команд:

view = DetailView()
html = view.render_request({'url': 'https://stepik.org/course/116336/', 'method': 'PUT'})
print(html)
будет выведено:

PUT: https://stepik.org/course/116336/

Это и есть принцип работы паттерна миксинов.

P.S. В программе требуется объявить только класс GeneralView. На экран выводить ничего не нужно.